# LeWorldModel (LeWM) 复现 Notebook

完整复现 LeWM 训练流程，**兼容 Mac（MPS/CPU）和 Google Colab（T4 GPU）**。

| 环境 | `MAX_EPOCHS` | `BATCH_SIZE` | 预计时间 |
|---|---|---|---|
| Mac 本地验证 | 1 | 32 | ~5-15 分钟 |
| Colab T4 完整训练 | 100 | 128 | ~3-5 小时 |

**使用方法**：Mac 本地在 LeWM 项目根目录激活 venv 后直接运行；Colab 见 Section 0。

In [1]:
import sys
import os

IN_COLAB = 'google.colab' in sys.modules
IS_MAC   = sys.platform == 'darwin'

env_name = 'Google Colab' if IN_COLAB else ('Mac' if IS_MAC else 'Linux')
print(f'环境：{env_name}  |  Python {sys.version.split()[0]}')

环境：Google Colab  |  Python 3.12.13


## Section 0：Colab 环境安装

**仅在 Google Colab 中**取消注释并执行以下单元，Mac 本地跳过。

In [2]:
# ===== 仅在 Google Colab 中取消注释 =====

# # 1. 克隆仓库
# !git clone https://github.com/lucas-maes/le-wm.git
# %cd le-wm

# # 2. 安装依赖（训练不需要 env extra）
# !pip install -q "stable-worldmodel[train]"

# # 3. 验证
# for pkg in ['stable_pretraining', 'stable_worldmodel', 'lightning']:
#     try:
#         __import__(pkg)
#         print(f'OK  {pkg}')
#     except ImportError:
#         print(f'FAIL  {pkg}')

## Section 1：导入与设备配置

In [2]:
import json
from pathlib import Path

import torch
import torch.nn.functional as F
import lightning as pl
import stable_pretraining as spt
import stable_worldmodel as swm
import matplotlib.pyplot as plt
%matplotlib inline

# 从项目本地文件导入（jepa.py / module.py / utils.py）
from jepa import JEPA
from module import ARPredictor, Embedder, MLP, SIGReg
from utils import get_column_normalizer, get_img_preprocessor

print(f'torch      {torch.__version__}')
print(f'lightning  {pl.__version__}')

ModuleNotFoundError: No module named 'lightning'

In [4]:
# ─────────────────────────────────────────────────────────
# 设备自动检测：CUDA > MPS (Apple Silicon) > CPU
# ─────────────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE      = 'cuda'
    ACCELERATOR = 'gpu'
    PRECISION   = 'bf16-mixed'
elif torch.backends.mps.is_available():
    DEVICE      = 'mps'
    ACCELERATOR = 'mps'
    PRECISION   = '16-mixed'
else:
    DEVICE      = 'cpu'
    ACCELERATOR = 'cpu'
    PRECISION   = '16-mixed'

print(f'设备：{DEVICE}  |  加速器：{ACCELERATOR}  |  精度：{PRECISION}')

设备：mps  |  加速器：mps  |  精度：16-mixed


## Section 2：训练超参

替代原项目的 Hydra yaml 配置。  
- **Mac 验证**：保持默认（`MAX_EPOCHS=1`，`BATCH_SIZE=32`）  
- **Colab 完整训练**：改为 `MAX_EPOCHS=100`，`BATCH_SIZE=128`

In [5]:
# ── 根据目标环境修改这两行 ───────────────────────────────
MAX_EPOCHS = 1    # Mac 验证：1 ；Colab 完整训练：100
BATCH_SIZE = 32   # Mac：32 ；Colab T4：128

# ── 固定超参（与原论文保持一致）────────────────────────────
IMG_SIZE      = 224
EMBED_DIM     = 192
HISTORY_SIZE  = 3
NUM_PREDS     = 1
NUM_STEPS     = HISTORY_SIZE + NUM_PREDS  # 每条样本的帧数 = 4
FRAMESKIP     = 5   # pusht：5 帧堆叠为 1 步

SIGREG_WEIGHT = 0.09
LR            = 5e-5
WEIGHT_DECAY  = 1e-3
GRAD_CLIP     = 1.0
TRAIN_SPLIT   = 0.9
SEED          = 3072

# Mac 多进程 fork 安全性限制，Colab 可用更多 worker
NUM_WORKERS   = 0 if IS_MAC else 2

print('超参配置完成')
print(f'  轮次：{MAX_EPOCHS}  批大小：{BATCH_SIZE}  设备：{DEVICE}')
print(f'  history_size={HISTORY_SIZE}  num_preds={NUM_PREDS}  frameskip={FRAMESKIP}')

超参配置完成
  轮次：1  批大小：32  设备：mps
  history_size=3  num_preds=1  frameskip=5


## Section 3：数据准备

数据集格式为 LanceDB（`.lance`），从 [HuggingFace 集合](https://huggingface.co/collections/quentinll/lewm) 下载。

下载后用 `tar --zstd -xvf <archive>.tar.zst` 解压到 `$STABLEWM_HOME`（默认 `~/.stable-wm/`）。

In [6]:
# STABLEWM_HOME：数据集和 checkpoint 的根目录
"""
用 wget 断点续传：

wget -c "https://hf-mirror.com/datasets/quentinll/lewm-pusht/resolve/main/pusht_expert_train.h5.zst" \
  -O ./data/pusht_expert_train.h5.zst \
  --header="Authorization: Bearer YOUR_HF_TOKEN" \
  --progress=bar:force

"""

if IN_COLAB:
    STABLEWM_HOME = os.environ.get('STABLEWM_HOME', os.path.expanduser('~/.stable-wm'))
else:
    STABLEWM_HOME = os.environ.get('STABLEWM_HOME', os.path.abspath('./data'))

os.makedirs(STABLEWM_HOME, exist_ok=True)
os.environ['STABLEWM_HOME'] = STABLEWM_HOME
print(f'STABLEWM_HOME = {STABLEWM_HOME}')

DATASET_NAME = 'pusht_expert_train.h5'
dataset_path = os.path.join(STABLEWM_HOME, DATASET_NAME)

if os.path.exists(dataset_path):
    print(f'数据集已就绪：{dataset_path}')
else:
    print(f'数据集不存在，请下载到 {STABLEWM_HOME}')

STABLEWM_HOME = /Users/ziyin/Workspace/GithubProjects/Buerduo/LeWM/data
数据集已就绪：/Users/ziyin/Workspace/GithubProjects/Buerduo/LeWM/data/pusht_expert_train.h5


In [7]:
# ── 注册 HDF5 格式（swm 默认未导入）─────────────────────────
from stable_worldmodel.data.formats.hdf5 import HDF5Dataset

# ── 加载数据集 ─────────────────────────────────────────────
DATASET_NAME = 'pusht_expert_train.h5'
dataset = HDF5Dataset(
    path=os.path.join(STABLEWM_HOME, DATASET_NAME),
    num_steps=NUM_STEPS,
    frameskip=FRAMESKIP,
    keys_to_load=['pixels', 'action', 'proprio', 'state'],
    keys_to_cache=['action', 'proprio', 'state'],
)
print(f'数据集大小：{len(dataset)} 条')

# ── action encoder 输入维度（运行时动态确定，不写在 yaml 里）─
ACTION_INPUT_DIM = FRAMESKIP * dataset.get_dim('action')
print(f'action_input_dim = {FRAMESKIP} × {dataset.get_dim("action")} = {ACTION_INPUT_DIM}')

# ── 数据变换管道：图像预处理 + 非图像列 Z-score 归一化 ──────
transforms_list = [
    get_img_preprocessor(source='pixels', target='pixels', img_size=IMG_SIZE)
]
for col in ['action', 'proprio', 'state']:
    transforms_list.append(get_column_normalizer(dataset, col, col))

dataset.transform = spt.data.transforms.Compose(*transforms_list)
print('变换管道构建完成')

15:56:04 | INFO  | __init__.py | Cached 'action' from '/Users/ziyin/Workspace/GithubProjects/Buerduo/LeWM/data/pusht_expert_train.h5'
15:56:04 | INFO  | __init__.py | Cached 'proprio' from '/Users/ziyin/Workspace/GithubProjects/Buerduo/LeWM/data/pusht_expert_train.h5'
15:56:04 | INFO  | __init__.py | Cached 'state' from '/Users/ziyin/Workspace/GithubProjects/Buerduo/LeWM/data/pusht_expert_train.h5'
数据集大小：1981721 条
action_input_dim = 5 × 2 = 10
变换管道构建完成


In [8]:
# ── Train / Val 划分 + DataLoader ──────────────────────────
g = torch.Generator().manual_seed(SEED)
train_size = int(TRAIN_SPLIT * len(dataset))
val_size   = len(dataset) - train_size
train_set, val_set = torch.utils.data.random_split(
    dataset, [train_size, val_size], generator=g
)

loader_kwargs = dict(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    persistent_workers=(NUM_WORKERS > 0),
    pin_memory=(DEVICE == 'cuda'),
)
if NUM_WORKERS > 0:
    loader_kwargs['prefetch_factor'] = 2

train_loader = torch.utils.data.DataLoader(
    train_set, shuffle=True, drop_last=True, generator=g, **loader_kwargs
)
val_loader = torch.utils.data.DataLoader(
    val_set, shuffle=False, drop_last=False, **loader_kwargs
)

print(f'训练集：{len(train_set):,}  验证集：{len(val_set):,}')
print(f'训练批次数：{len(train_loader)}')

训练集：1,783,548  验证集：198,173
训练批次数：55735


## Section 4：模型构建

LeWM 四大组件：**Encoder** → **Projector** → **ARPredictor**（AdaLN-zero）← **Action Encoder** → **Pred Projector**

In [9]:
torch.manual_seed(SEED)

# 1. ViT-Tiny Encoder（不使用预训练权重，patch=14×14）
encoder = spt.backbone.utils.vit_hf(
    'tiny', patch_size=14, image_size=IMG_SIZE,
    pretrained=False, use_mask_token=False,
)

# 2. Projector：MLP + BatchNorm1d  →  encoder 输出映射到 latent
projector = MLP(
    input_dim=EMBED_DIM, hidden_dim=2048, output_dim=EMBED_DIM,
    norm_fn=torch.nn.BatchNorm1d,
)

# 3. ARPredictor：因果 Transformer，用 AdaLN-zero 以 action 为条件
predictor = ARPredictor(
    num_frames=HISTORY_SIZE,
    input_dim=EMBED_DIM, hidden_dim=EMBED_DIM, output_dim=EMBED_DIM,
    depth=6, heads=16, mlp_dim=2048, dim_head=64,
    dropout=0.1, emb_dropout=0.0,
)

# 4. Action Encoder：Conv1d + MLP  →  action 序列编码
action_encoder = Embedder(input_dim=ACTION_INPUT_DIM, emb_dim=EMBED_DIM)

# 5. Pred Projector：与 Projector 结构相同，作用于预测嵌入
pred_proj = MLP(
    input_dim=EMBED_DIM, hidden_dim=2048, output_dim=EMBED_DIM,
    norm_fn=torch.nn.BatchNorm1d,
)

# 组装 JEPA 世界模型
model = JEPA(
    encoder=encoder, predictor=predictor, action_encoder=action_encoder,
    projector=projector, pred_proj=pred_proj,
)

print('模型构建完成')

15:56:11 | INFO  | utils.py    | Created ViT-tiny from scratch with config: {'hidden_size': 192, 'num_hidden_layers': 12, 'num_attention_heads': 3, 'intermediate_size': 768, 'image_size': 224, 'patch_size': 14}
模型构建完成


In [10]:
total = sum(p.numel() for p in model.parameters())
print(f'总参数量：{total:,}  ({total/1e6:.1f}M)')
print()
for name, mod in [
    ('encoder',        encoder),
    ('projector',      projector),
    ('predictor',      predictor),
    ('action_encoder', action_encoder),
    ('pred_proj',      pred_proj),
]:
    n = sum(p.numel() for p in mod.parameters())
    print(f'  {name:<16}  {n:>9,}  ({n/1e6:.2f}M)')

总参数量：18,034,478  (18.0M)

  encoder           5,501,376  (5.50M)
  projector           792,768  (0.79M)
  predictor         10,791,360  (10.79M)
  action_encoder      156,206  (0.16M)
  pred_proj           792,768  (0.79M)


## Section 5：单批次前向验证

训练前用一个批次验证完整数据流，确认 tensor 形状和损失值正常。

In [11]:
model_dev = model.eval().to(DEVICE)

sample = next(iter(train_loader))
sample = {k: v.to(DEVICE) if torch.is_tensor(v) else v for k, v in sample.items()}

with torch.no_grad():
    # encode：pixels → CLS token → projector → emb
    out     = model_dev.encode(sample)
    emb     = out['emb']       # (B, T=4, D=192)
    act_emb = out['act_emb']   # (B, T=4, D=192)

    # predict：取前 history_size 步，预测后 1 步
    ctx_emb  = emb[:, :HISTORY_SIZE]      # (B, 3, 192)
    ctx_act  = act_emb[:, :HISTORY_SIZE]
    tgt_emb  = emb[:, NUM_PREDS:]         # (B, 3, 192) — 目标（偏移 1 步）
    pred_emb = model_dev.predict(ctx_emb, ctx_act)  # (B, 3, 192)

    # losses
    pred_loss   = (pred_emb - tgt_emb).pow(2).mean()
    sigreg_test = SIGReg().to(DEVICE)
    sig_loss    = sigreg_test(emb.transpose(0, 1))

print('前向传播验证通过')
print(f'  emb      {tuple(emb.shape)}')
print(f'  pred_emb {tuple(pred_emb.shape)}')
print(f'  tgt_emb  {tuple(tgt_emb.shape)}')
print(f'  pred_loss   = {pred_loss.item():.4f}')
print(f'  sigreg_loss = {sig_loss.item():.4f}')

前向传播验证通过
  emb      (32, 4, 192)
  pred_emb (32, 3, 192)
  tgt_emb  (32, 3, 192)
  pred_loss   = 0.0774
  sigreg_loss = 14.0314


## Section 6：训练

`LeWMModule` 实现 `train.py:lejepa_forward` 的完整逻辑，  
Loss = `pred_loss + 0.09 × sigreg_loss`。

In [16]:
class LeWMModule(pl.LightningModule):
    """复现 train.py:lejepa_forward 逻辑的 Lightning 训练模块。"""

    def __init__(self, model, history_size, num_preds, sigreg_weight,
                 lr, weight_decay, max_epochs):
        super().__init__()
        self.model         = model
        self.sigreg        = SIGReg(knots=17, num_proj=1024)
        self.history_size  = history_size
        self.num_preds     = num_preds
        self.sigreg_weight = sigreg_weight
        self.lr            = lr
        self.weight_decay  = weight_decay
        self.max_epochs    = max_epochs

    def _step(self, batch):
        batch['action'] = torch.nan_to_num(batch['action'], 0.0)
        out = self.model.encode(batch)
        emb     = out['emb']      # (B, T, D)
        act_emb = out['act_emb']

        ctx_emb  = emb[:, :self.history_size]
        ctx_act  = act_emb[:, :self.history_size]
        tgt_emb  = emb[:, self.num_preds:]
        pred_emb = self.model.predict(ctx_emb, ctx_act)

        pred_loss   = (pred_emb - tgt_emb).pow(2).mean()
        sigreg_loss = self.sigreg(emb.transpose(0, 1))
        loss        = pred_loss + self.sigreg_weight * sigreg_loss
        return loss, pred_loss, sigreg_loss

    def training_step(self, batch, batch_idx):
        loss, pred_loss, sigreg_loss = self._step(batch)
        self.log_dict({
            'train/loss': loss, 'train/pred_loss': pred_loss,
            'train/sigreg_loss': sigreg_loss,
        }, on_step=True, prog_bar=True, sync_dist=True)
        return loss

    def validation_step(self, batch, batch_idx):
        loss, pred_loss, sigreg_loss = self._step(batch)
        self.log_dict({
            'val/loss': loss, 'val/pred_loss': pred_loss,
            'val/sigreg_loss': sigreg_loss,
        }, on_epoch=True, prog_bar=True, sync_dist=True)
        return loss

    def configure_optimizers(self):
        opt = torch.optim.AdamW(
            self.model.parameters(), lr=self.lr, weight_decay=self.weight_decay
        )
        # LinearWarmup (10%) + CosineAnnealing，与原论文一致
        warmup = max(1, int(0.1 * self.max_epochs))
        sched = torch.optim.lr_scheduler.SequentialLR(
            opt,
            schedulers=[
                torch.optim.lr_scheduler.LinearLR(
                    opt, start_factor=0.1, end_factor=1.0, total_iters=warmup
                ),
                torch.optim.lr_scheduler.CosineAnnealingLR(
                    opt, T_max=max(1, self.max_epochs - warmup)
                ),
            ],
            milestones=[warmup],
        )
        return {'optimizer': opt, 'lr_scheduler': {'scheduler': sched, 'interval': 'epoch'}}

In [17]:
class LossHistory(pl.Callback):
    """记录每个 step 的训练指标和每个 epoch 的验证指标。"""

    def __init__(self):
        # step-level train metrics
        self.steps         = []
        self.train_loss    = []
        self.train_pred    = []
        self.train_sigreg  = []
        # epoch-level val metrics
        self.epochs        = []
        self.val_loss      = []
        self.val_pred      = []
        self.val_sigreg    = []
        # learning rate per epoch
        self.lr_history    = []

    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        m = trainer.callback_metrics
        self.steps.append(trainer.global_step)
        self.train_loss.append(_get(m, 'train/loss'))
        self.train_pred.append(_get(m, 'train/pred_loss'))
        self.train_sigreg.append(_get(m, 'train/sigreg_loss'))

    def on_validation_epoch_end(self, trainer, pl_module):
        m = trainer.callback_metrics
        if 'val/loss' not in m:
            return
        self.epochs.append(trainer.current_epoch)
        self.val_loss.append(_get(m, 'val/loss'))
        self.val_pred.append(_get(m, 'val/pred_loss'))
        self.val_sigreg.append(_get(m, 'val/sigreg_loss'))
        # 当前学习率
        opt = trainer.optimizers[0]
        self.lr_history.append(opt.param_groups[0]['lr'])

def _get(metrics, key):
    v = metrics.get(key)
    return float(v.detach().cpu()) if v is not None else float('nan')

In [18]:
from lightning.pytorch.callbacks import TQDMProgressBar
from lightning.pytorch.loggers import WandbLogger

# 重置模型到 CPU（Lightning 会自动迁移到目标设备）
model = model.cpu().train()

lewm = LeWMModule(
    model=model,
    history_size=HISTORY_SIZE,
    num_preds=NUM_PREDS,
    sigreg_weight=SIGREG_WEIGHT,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    max_epochs=MAX_EPOCHS,
)

loss_history = LossHistory()
progress_bar = TQDMProgressBar(refresh_rate=10)

# entity=None 自动使用登录账号的默认空间，无需手动填写
wandb_logger = WandbLogger(
    entity=None,
    project='lewm',
    name=f'pusht-ep{MAX_EPOCHS}-bs{BATCH_SIZE}',
)

trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator=ACCELERATOR,
    devices=1,
    precision=PRECISION,
    gradient_clip_val=GRAD_CLIP,
    callbacks=[loss_history, progress_bar],
    logger=wandb_logger,
    enable_checkpointing=False,
    num_sanity_val_steps=1,
    enable_progress_bar=True,
)

print(f'开始训练：{MAX_EPOCHS} epoch，加速器={ACCELERATOR}，精度={PRECISION}')
trainer.fit(lewm, train_loader, val_loader)
print('训练完成')

Using 16bit Automatic Mixed Precision (AMP)


16:01:22 | INFO  | env_info.py |   EnvironmentDumpCallback initialized (filename=environment.json, async=True)
16:01:22 | INFO  | utils.py    | ── HuggingFaceCheckpoint ─────────────────────────
16:01:22 | INFO  | hf_models.py|   save_dir: <cyan>hf_exports</cyan>
16:01:22 | INFO  | hf_models.py|   per_step: <cyan>False</cyan>  (False ⇒ overwrite single 'last/' subdir; True ⇒ keep step_N/)


Adding 14 callbacks from entry point 'stablepretraining_callbacks': PrintProgressBar, ModuleRegistryCallback, LoggingCallback, EnvironmentDumpCallback, TrainerInfo, SklearnCheckpoint, WandbCheckpoint, TrackioCheckpoint, SwanLabCheckpoint, ModuleSummary, SLURMInfo, LogUnusedParametersOnce, HuggingFaceCheckpointCallback, HardwareMonitor
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


开始训练：1 epoch，加速器=mps，精度=16-mixed


16:01:24 | INFO  | utils.py    | ── EnvironmentDump ───────────────────────────────
16:01:24 | INFO  | env_info.py |   log_dir: /Users/ziyin/.cache/stable-pretraining
16:01:24 | INFO  | env_info.py |   Running environment dump in background thread (non-blocking)
16:01:24 | INFO  | env_info.py |   Collecting environment information...
16:01:24 | OK    | env_info.py | ✓ Background thread started successfully
16:01:24 | OK    | env_info.py | ✓ Python 3.10.18
16:01:24 | INFO  | trainer_inf~|   linking trainer to DataModule
16:01:24 | OK    | env_info.py | ✓ System: Darwin 24.6.0 (arm64)
16:01:24 | INFO  | utils.py    | ── Callbacks (in order) ──────────────────────────
16:01:24 | INFO  | env_info.py |   Hostname: Ziyin-MBP.local
16:01:24 | INFO  | utils.py    |     [ 0] LossHistory
16:01:24 | INFO  | utils.py    |     [ 1] TQDMProgressBar
16:01:24 | INFO  | utils.py    |     [ 2] RichModelSummary
16:01:24 | INFO  | utils.py    |     [ 3] PrintProgressBar
16:01:24 | INFO  | utils.py    |   

...

Training: |          | 0/? [00:00<?, ?it/s]

16:01:26 | WARN  | hf_models.py| ! No Hugging Face (PreTrainedModel) submodules found in LightningModule.
16:01:26 | INFO  | utils.py    | ── LogUnusedParametersOnce ───────────────────────
16:01:26 | INFO  | unused_para~|   registered hooks on 297 leaf parameters
16:01:31 | OK    | unused_para~| ✓ all tracked parameters received gradients on the first backward pass
16:01:31 | INFO  | unused_para~|   hooks removed, callback disabled
[Epoch 0/1] step 50/55735 (1.1 it/s) | train/loss: 0.8871 | train/pred_loss: 0.2616 | train/sigreg_loss: 6.949
[Epoch 0/1] step 100/55735 (1.2 it/s) | train/loss: 0.7739 | train/pred_loss: 0.2573 | train/sigreg_loss: 5.742
[Epoch 0/1] step 150/55735 (1.2 it/s) | train/loss: 0.7129 | train/pred_loss: 0.25 | train/sigreg_loss: 5.145
[Epoch 0/1] step 200/55735 (1.2 it/s) | train/loss: 0.674 | train/pred_loss: 0.2411 | train/sigreg_loss: 4.809
[Epoch 0/1] step 250/55735 (1.2 it/s) | train/loss: 0.6435 | train/pred_loss: 0.2074 | train/sigreg_loss: 4.844
[Epoch 


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

/Users/ziyin/Workspace/GithubProjects/Buerduo/LeWM/.venv/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
import numpy as np

def smooth(vals, w=20):
    """简单滑动平均平滑曲线。"""
    if len(vals) < w:
        return vals
    return np.convolve(vals, np.ones(w)/w, mode='valid')

steps  = loss_history.steps
epochs = loss_history.epochs if loss_history.epochs else list(range(1, len(loss_history.val_loss)+1))

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
fig.suptitle('LeWM Training Dashboard', fontsize=14, fontweight='bold')

# ── 辅助函数 ─────────────────────────────────────────────
def plot_train(ax, vals, title, color='steelblue'):
    if not vals:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(title); return
    ax.plot(steps, vals, alpha=0.25, color=color, linewidth=0.6, label='raw')
    s = smooth(vals)
    ax.plot(steps[len(steps)-len(s):], s, color=color, linewidth=1.5, label='smooth')
    ax.set_xlabel('Step'); ax.set_title(title); ax.grid(alpha=0.3); ax.legend(fontsize=8)

def plot_val(ax, vals, title, color='tomato', marker='o'):
    if not vals:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(title); return
    ax.plot(epochs, vals, marker=marker, color=color, linewidth=2, markersize=5)
    ax.set_xlabel('Epoch'); ax.set_title(title); ax.grid(alpha=0.3)

# ── Row 0：总 Loss ───────────────────────────────────────
plot_train(axes[0,0], loss_history.train_loss,   'Train Loss (total)')
plot_val  (axes[0,1], loss_history.val_loss,     'Val Loss (total)')

# Train vs Val 总 Loss 叠加（epoch 级别）
ax = axes[0,2]
if loss_history.val_loss:
    # 把 train_loss 按 epoch 平均后叠加
    ep_boundaries = np.array_split(loss_history.train_loss, max(1, len(epochs)))
    train_ep = [np.nanmean(b) for b in ep_boundaries if len(b)]
    ep_x = list(range(1, len(train_ep)+1))
    ax.plot(ep_x,  train_ep,           'o-', color='steelblue', label='train', linewidth=1.5, markersize=4)
    ax.plot(epochs, loss_history.val_loss, 's-', color='tomato',    label='val',   linewidth=1.5, markersize=4)
    ax.set_xlabel('Epoch'); ax.legend(fontsize=9)
else:
    ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
ax.set_title('Train vs Val Loss'); ax.grid(alpha=0.3)

# ── Row 1：Pred Loss ─────────────────────────────────────
plot_train(axes[1,0], loss_history.train_pred,  'Train Pred Loss', color='mediumseagreen')
plot_val  (axes[1,1], loss_history.val_pred,    'Val Pred Loss',   color='darkorange')

# Pred Loss 趋势
ax = axes[1,2]
if loss_history.val_pred:
    ep_boundaries = np.array_split(loss_history.train_pred, max(1, len(epochs)))
    train_ep = [np.nanmean(b) for b in ep_boundaries if len(b)]
    ep_x = list(range(1, len(train_ep)+1))
    ax.plot(ep_x,   train_ep,                'o-', color='mediumseagreen', label='train', linewidth=1.5, markersize=4)
    ax.plot(epochs, loss_history.val_pred,   's-', color='darkorange',     label='val',   linewidth=1.5, markersize=4)
    ax.set_xlabel('Epoch'); ax.legend(fontsize=9)
else:
    ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
ax.set_title('Train vs Val Pred Loss'); ax.grid(alpha=0.3)

# ── Row 2：SIGReg Loss / LR / 分布 ──────────────────────
plot_train(axes[2,0], loss_history.train_sigreg, 'Train SIGReg Loss', color='mediumpurple')
plot_val  (axes[2,1], loss_history.val_sigreg,   'Val SIGReg Loss',   color='slateblue')

# 学习率曲线
ax = axes[2,2]
if loss_history.lr_history:
    ax.plot(epochs, loss_history.lr_history, color='goldenrod', linewidth=2, marker='.')
    ax.set_xlabel('Epoch'); ax.set_ylabel('LR'); ax.set_title('Learning Rate Schedule')
    ax.ticklabel_format(style='sci', axis='y', scilimits=(0,0))
    ax.grid(alpha=0.3)
else:
    ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('Learning Rate Schedule')

plt.tight_layout()
plt.savefig('training_dashboard.png', dpi=300, bbox_inches='tight')
plt.show()
print('已保存 training_dashboard.png (300 dpi)')

## Section 7：Checkpoint 保存与验证

保存两种格式（与原项目完全兼容）：
- `lewm_object.ckpt`：完整 Python 对象，`eval.py` 和 `swm.policy.AutoCostModel` 使用
- `weights.pt`：纯 state dict，HuggingFace 发布格式

In [ ]:
ckpt_dir = Path(STABLEWM_HOME) / 'checkpoints' / 'notebook_run'
ckpt_dir.mkdir(parents=True, exist_ok=True)

# 1. 完整 Python 对象（eval.py 直接加载）
object_path = ckpt_dir / 'lewm_object.ckpt'
torch.save(lewm.model.cpu(), object_path)
print(f'完整对象已保存：{object_path}')

# 2. 纯权重 state dict
weights_path = ckpt_dir / 'weights.pt'
torch.save(lewm.model.state_dict(), weights_path)
print(f'权重已保存：    {weights_path}')

# 3. 模型配置 JSON（与 HuggingFace 格式兼容）
config_dict = {
    'encoder':        {'size': 'tiny', 'patch_size': 14, 'image_size': IMG_SIZE},
    'predictor':      {'num_frames': HISTORY_SIZE, 'input_dim': EMBED_DIM,
                       'hidden_dim': EMBED_DIM, 'output_dim': EMBED_DIM,
                       'depth': 6, 'heads': 16, 'mlp_dim': 2048, 'dim_head': 64},
    'action_encoder': {'input_dim': ACTION_INPUT_DIM, 'emb_dim': EMBED_DIM},
    'projector':      {'input_dim': EMBED_DIM, 'hidden_dim': 2048, 'output_dim': EMBED_DIM},
    'pred_proj':      {'input_dim': EMBED_DIM, 'hidden_dim': 2048, 'output_dim': EMBED_DIM},
}
with open(ckpt_dir / 'config.json', 'w') as f:
    json.dump(config_dict, f, indent=2)
print(f'配置已保存：    {ckpt_dir / "config.json"}')

In [ ]:
# 加载验证：从 _object.ckpt 恢复模型，执行一次前向传播
loaded = torch.load(object_path, map_location='cpu', weights_only=False)
loaded.eval()

test_batch = {k: v[:2] if torch.is_tensor(v) else v
              for k, v in next(iter(val_loader)).items()}

with torch.no_grad():
    result = loaded.encode(test_batch)
    print(f'加载验证通过  emb 形状：{tuple(result["emb"].shape)}')

print()
print('要在 eval.py 中使用此 checkpoint，执行：')
print(f'  python eval.py --config-name=pusht.yaml policy=checkpoints/notebook_run/lewm')

## Section 8：升级至 Colab 完整训练

1. 上传此 notebook 到 Google Colab，选择 **T4 GPU** 运行时
2. 取消注释 **Section 0** 安装单元
3. 在 Section 2 修改两个参数：
   ```python
   MAX_EPOCHS = 100
   BATCH_SIZE = 128
   ```
4. （可选）启用 WandB：在 Section 6 训练执行单元中替换：
   ```python
   # 取消注释这两行：
   from lightning.pytorch.loggers import WandbLogger
   wandb_logger = WandbLogger(entity='your_entity', project='lewm')
   # 将 logger=False 改为：
   logger=wandb_logger
   ```
5. **Runtime → Run all** 顺序执行所有单元

**Colab T4 参考时间**：pusht 数据集 100 epoch 约 3–5 小时  
**Checkpoint 位置**：`$STABLEWM_HOME/checkpoints/notebook_run/`